<a href="https://colab.research.google.com/github/Brummelmayano/Compteur-intelligent/blob/main/model/code/TFlite_ssd_mobilenet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.&nbsp;Rassembler et étiqueter les images d'entrainement


# 2.&nbsp;Installer l'objet TensorFlow


Tout d'abord, nous allons installer l'API de détection d'objets TensorFlow dans cette instance Google Colab. Cela nécessite le clonage du [dépôt de modèles TensorFlow](https://github.com/tensorflow/models) et l'exécution de quelques commandes d'installation.

In [ ]:
# Cloner le référentiel de modèles Tensorflow depuis GitHub

!pip uninstall Cython -y # Correction temporaire de l'erreur "No module named 'object_detection'"
!git clone --depth 1 https://github.com/tensorflow/models

Found existing installation: Cython 3.0.6
Uninstalling Cython-3.0.6:
  Successfully uninstalled Cython-3.0.6
Cloning into 'models'...
remote: Enumerating objects: 4065, done.
remote: Counting objects: 100% (4065/4065), done.
remote: Compressing objects: 100% (3105/3105), done.
remote: Total 4065 (delta 1184), reused 1938 (delta 900), pack-reused 0
Receiving objects: 100% (4065/4065), 56.43 MiB | 29.32 MiB/s, done.
Resolving deltas: 100% (1184/1184), done.


In [ ]:
# Copiez les fichiers d'installation dans le dossier models/research

%%bash
cd models/research/
protoc object_detection/protos/*.proto --python_out=.
#cp object_detection/packages/tf2/setup.py .

In [ ]:
# Modifier le fichier setup.py pour installer le référentiel tf-models-official ciblé sur TF v2.8.0

import re
with open('/content/models/research/object_detection/packages/tf2/setup.py') as f:
    s = f.read()

with open('/content/models/research/setup.py', 'w') as f:
    # Set fine_tune_checkpoint path
    s = re.sub('tf-models-official>=2.5.1',
               'tf-models-official==2.8.0', s)
    f.write(s)

In [ ]:
# Installez l'API de détection d'objets (REMARQUE : l'exécution de ce bloc prend environ 10 minutes)

# Besoin d'effectuer un correctif temporaire avec PyYAML car Colab ne parvient pas à installer PyYAML v5.4.1

!pip install pyyaml==5.3
!pip install /content/models/research/

# Nécessité de rétrograder vers TF v2.8.0 en raison d'un bug de compatibilité Colab avec TF v2.10 (au 03/10/22)
!pip install tensorflow==2.8.0

# Installez CUDA version 11.0 (pour maintenir la compatibilité avec TF v2.8.0)

!pip install tensorflow_io==0.23.1
!wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64/cuda-ubuntu1804.pin
!mv cuda-ubuntu1804.pin /etc/apt/preferences.d/cuda-repository-pin-600
!wget http://developer.download.nvidia.com/compute/cuda/11.0.2/local_installers/cuda-repo-ubuntu1804-11-0-local_11.0.2-450.51.05-1_amd64.deb
!dpkg -i cuda-repo-ubuntu1804-11-0-local_11.0.2-450.51.05-1_amd64.deb
!apt-key add /var/cuda-repo-ubuntu1804-11-0-local/7fa2af80.pub
!apt-get update && sudo apt-get install cuda-toolkit-11-0
!export LD_LIBRARY_PATH=/usr/local/cuda-11.0/lib64:$LD_LIBRARY_PATH

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.2/268.2 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyyaml: filename=PyYAML-5.3-cp310-cp310-linux_x86_64.whl size=44244 sha256=902b11ceab62fac2c4d76758d86707a6685eaa8de09f287f744de683fe791ec0
  Stored in directory: /root/.cache/pip/wheels/0d/72/68/a263cfc14175636cf26bada99f13b735be1b60a11318e08bfc
Successfully built pyyaml
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0.1
    Uninstalling PyYAML-6.0.1:
      Successfully uninstalled PyYAML-6.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llmx 0.0.15a0 requires cohere, which is not installed.
llmx 0.0.15a0 requires openai, which is not installed.
llmx 0.0.15a0 requires tiktoken, which is not installed.
dask 2023.8.1 requires pyyaml>=5.3.1, but you have pyyaml 5.3 which is incompatible.
distr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 497.6/497.6 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.5/462.5 kB 37.1 MB/s eta 0:00:00
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.8.1
    Uninstalling tensorflow-2.8.1:
      Successfully uninstalled tensorflow-2.8.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 33.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 73.1 MB/s eta 0:00:00
  Attempting uninstall: tensorflow-io-gcs-filesystem
    Found existing installation: tensorflow-io-gcs-filesystem 0.35.0
    Uninstalling tensorflow-io-gcs-filesystem-0.35.0:
      Successfully uninstalled tensorflow-io-gcs-filesystem-0.35.0
  Attempting uninstall: tensorflow_io
    Found existing installation: tensorflow-io 0.35.0
    Uninstalling tensorflow-io-0.35.0:
      Successfully uninstalled tensorflow-io-0.35.0
--2024-01-11 06:24:58--  https://developer.download.nvidia.com/compute/cuda/repos

Testons notre installation en exécutant `model_builder_tf2_test.py` pour nous assurer que tout fonctionne comme prévu.

In [ ]:
!python /content/models/research/object_detection/builders/model_builder_tf2_test.py


2024-01-11 06:29:54.073516: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64
2024-01-11 06:29:54.073598: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
Running tests under Python 3.10.12: /usr/bin/python3
[ RUN      ] ModelBuilderTF2Test.test_create_center_net_deepmac
W0111 06:29:55.288138 136782815690752 model_builder.py:1112] Building experimental DeepMAC meta-arch. Some features may be omitted.
INFO:tensorflow:time(__main__.ModelBuilderTF2Test.test_create_center_net_deepmac): 2.14s
I0111 06:29:56.219733 136782815690752 test_util.py:2373] time(__main__.ModelBuilderTF2Test.test_create_center_net_deepmac): 2.14s
[       OK ] ModelBuilderTF2Test.test_create_center_net_deepmac
[ RUN      ] ModelBuilderTF2Test.test_create_center_net_mod

# 3.&nbsp;Télécharger l'ensemble de données d'image et préparer les données d'entraînement

Dans cette section, nous allons télécharger nos données et les préparer pour l'entraînement avec TensorFlow. Nous allons télécharger nos images, les diviser en dossiers d'entraînement, de validation et de test, puis exécuter des scripts pour créer des TFRecords à partir de nos données.
```
-- images
  -- img1.jpg
  -- img1.xml
  -- img2.jpg
  -- img2.xml
  ...
```

### 3.1 Télécharger des images depuis drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')



Mounted at /content/gdrive


In [ ]:
!mkdir /content/images
!mkdir /content/images/all; mkdir /content/images/train; mkdir /content/images/validation; mkdir /content/images/test
!cp /content/gdrive/MyDrive/Unikin/memoire/roidetection/scoreboard/* /content/images/all



Ensuite, nous diviserons les images en ensembles d’entraînement, de validation et de test. Voici à quoi sert chaque ensemble :



* **Train** : ce sont les images réelles utilisées pour entraîner le modèle. À chaque étape d'entrainement, un lot d’images du « train » est transmis au réseau neuronal. Le réseau prédit les classes et les emplacements des objets dans les images. L'algorithme d'entrainement calcule la perte (c'est-à-dire à quel point les prédictions étaient « fausses ») et ajuste les pondérations du réseau par rétropropagation.


* **Validation** : les images de l'ensemble "validation" peuvent être utilisées par l'algorithme d'entraînement pour vérifier la progression de l'entraînement et ajuster les hyperparamètres (comme le taux d'apprentissage). Contrairement aux images « d'entraînement », ces images ne sont utilisées que périodiquement pendant l'entraînement (c'est-à-dire une fois toutes les un certain nombre d'étapes d'entraînement).


* **Test** : Ces images ne sont jamais vues par le réseau neuronal pendant l'entraînement. Ils sont destinés à être utilisés par un humain pour effectuer les tests finaux du modèle afin de vérifier la précision du modèle.

on va utiliser un script Python pour déplacer aléatoirement 80 % des images vers le dossier « train », 10 % vers le dossier « validation » et 10 % vers le dossier « test ».

nous commençons par télécharger le script

In [ ]:
!wget https://raw.githubusercontent.com/EdjeElectronics/TensorFlow-Lite-Object-Detection-on-Android-and-Raspberry-Pi/master/util_scripts/train_val_test_split.py
!python train_val_test_split.py

--2024-01-11 06:31:24--  https://raw.githubusercontent.com/EdjeElectronics/TensorFlow-Lite-Object-Detection-on-Android-and-Raspberry-Pi/master/util_scripts/train_val_test_split.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2803 (2.7K) [text/plain]
Saving to: ‘train_val_test_split.py’

train_val_test_spli 100%[===================>]   2.74K  --.-KB/s    in 0s      

2024-01-11 06:31:24 (32.2 MB/s) - ‘train_val_test_split.py’ saved [2803/2803]

Total images: 171
Images moving to train: 136
Images moving to validation: 17
Images moving to test: 18


## 3.3 Créer des Labelmap et des TFRecords

Enfin, nous devons créer une carte d'étiquettes pour le détecteur et convertir les images dans un format de fichier de données appelé TFRecords, qui est utilisé par TensorFlow pour l'entrainement.
Nous utiliserons des scripts Python pour convertir automatiquement les données au format TFRecord.
Avant de les exécuter, nous devons définir un labelmap pour nos classes.

La section de code ci-dessous créera un fichier "labelmap.txt" contenant une liste de classes.

dans notre cas une seule classe qui est scoreboard   

In [ ]:
%%bash
cat <<EOF >> /content/labelmap.txt
scoreboard
EOF

Téléchargeons et exécutons les scripts de conversion de données à partir du [dépôt GitHub](https://github.com/EdjeElectronics/TensorFlow-Lite-Object-Detection-on-Android-and-Raspberry-Pi) en cliquant sur play dans les trois sections suivantes de code. Ils créeront des fichiers TFRecord pour les ensembles de données d'entraînement et de validation, ainsi qu'un fichier « labelmap.pbtxt » qui contient le labelmap dans un format différent.

In [ ]:
# Télécharger des scripts de conversion de données
! wget https://raw.githubusercontent.com/EdjeElectronics/TensorFlow-Lite-Object-Detection-on-Android-and-Raspberry-Pi/master/util_scripts/create_csv.py
! wget https://raw.githubusercontent.com/EdjeElectronics/TensorFlow-Lite-Object-Detection-on-Android-and-Raspberry-Pi/master/util_scripts/create_tfrecord.py

--2024-01-11 06:31:25--  https://raw.githubusercontent.com/EdjeElectronics/TensorFlow-Lite-Object-Detection-on-Android-and-Raspberry-Pi/master/util_scripts/create_csv.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1348 (1.3K) [text/plain]
Saving to: ‘create_csv.py’

create_csv.py       100%[===================>]   1.32K  --.-KB/s    in 0s      

2024-01-11 06:31:25 (64.8 MB/s) - ‘create_csv.py’ saved [1348/1348]

--2024-01-11 06:31:25--  https://raw.githubusercontent.com/EdjeElectronics/TensorFlow-Lite-Object-Detection-on-Android-and-Raspberry-Pi/master/util_scripts/create_tfrecord.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githu

In [ ]:
# Créer des fichiers de données CSV et des fichiers TFRecord
!python3 create_csv.py
!python3 create_tfrecord.py --csv_input=images/train_labels.csv --labelmap=labelmap.txt --image_dir=images/train --output_path=train.tfrecord
!python3 create_tfrecord.py --csv_input=images/validation_labels.csv --labelmap=labelmap.txt --image_dir=images/validation --output_path=val.tfrecord

Successfully converted xml to csv.
Successfully converted xml to csv.
2024-01-11 06:31:31.507152: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64
2024-01-11 06:31:31.507227: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
Successfully created the TFRecords: /content/train.tfrecord
2024-01-11 06:31:36.632056: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/nvidia/lib:/usr/local/nvidia/lib64
2024-01-11 06:31:36.632122: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
Successfully created the TFRecords: /content/val.tfr

Nous stockerons les emplacements des fichiers TFRecord et labelmap en tant que variables afin de pouvoir les référencer plus tard dans cette session Colab.

In [ ]:
train_record_fname = '/content/train.tfrecord'
val_record_fname = '/content/val.tfrecord'
label_map_pbtxt_fname = '/content/labelmap.pbtxt'

# 4.&nbsp;configuration de l'entraînement

Dans cette section, nous allons préparer le modèle et la configuration d'entraînement. Cela signifie que nous allons mettre en place les paramètres du modèle que nous allons utiliser ainsi que les configurations d'entraînement telles que le taux d'apprentissage, le nombre total d'étapes d'entraînement, etc.

Nous préciserons quel modèle TensorFlow pré-entraîné nous souhaitons utiliser à partir du [Zoo de modèles de détection d'objets TensorFlow 2](https://github.com/tensorflow/models/blob/master/research/object_detection/g3doc/tf2_detection_zoo.md). Chaque modèle est également livré avec un fichier de configuration qui pointe vers les emplacements des fichiers, définit les paramètres d'entrainement (tels que le taux d'apprentissage et le nombre total d'étapes d'entrainement), et bien plus encore. Nous allons modifier le fichier de configuration pour notre tâche d'entrainement personnalisée.

La première section de code répertorie certains modèles disponibles dans le zoo modèle TF2 et définit certains noms de fichiers qui seront utilisés ultérieurement pour télécharger le modèle et le fichier de configuration. Cela facilite la gestion du modèle que nous utilisons et l'ajout ultérieur d'autres modèles à la liste.

definir la variable "chosen_model" pour qu'elle corresponde au nom du modèle avec lequel nous souhaitons entraîner.
Dans notre cas le modele « ssd-mobilenet-v2 ».



https://github.com/tensorflow/models/blob/master/research/object_detection/g3doc/tf2_detection_zoo.md

In [ ]:
# Modifiez la variable selected_model pour déployer différents modèles disponibles dans le zoo de détection d'objets TF2

chosen_model = 'ssd-mobilenet-v2-fpnlite-320'

MODELS_CONFIG = {
    'ssd-mobilenet-v2': {
        'model_name': 'ssd_mobilenet_v2_320x320_coco17_tpu-8',
        'base_pipeline_file': 'ssd_mobilenet_v2_320x320_coco17_tpu-8.config',
        'pretrained_checkpoint': 'ssd_mobilenet_v2_320x320_coco17_tpu-8.tar.gz',
    },
    'efficientdet-d0': {
        'model_name': 'efficientdet_d0_coco17_tpu-32',
        'base_pipeline_file': 'ssd_efficientdet_d0_512x512_coco17_tpu-8.config',
        'pretrained_checkpoint': 'efficientdet_d0_coco17_tpu-32.tar.gz',
    },
    'ssd-mobilenet-v2-fpnlite-320': {
        'model_name': 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8',
        'base_pipeline_file': 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8.config',
        'pretrained_checkpoint': 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8.tar.gz',
    },
}

model_name = MODELS_CONFIG[chosen_model]['model_name']
pretrained_checkpoint = MODELS_CONFIG[chosen_model]['pretrained_checkpoint']
base_pipeline_file = MODELS_CONFIG[chosen_model]['base_pipeline_file']

Téléchargez le fichier de modèle pré-entraîné et le fichier de configuration

In [ ]:
# Créez un dossier "mymodel" pour contenir les poids pré-entraînés et les fichiers de configuration
"""
"Poids pré-entraînés" se réfère aux paramètres appris par un modèle lorsqu'il
est entraîné sur un ensemble de données, et ces poids peuvent être utilisés pour
initialiser un nouveau modèle ou pour la poursuite de l'entraînement."""

#%mkdir /content/models/mymodel/
%mkdir -p /content/gdrive/MyDrive/Unikin/memoire/roidetection/mymodel/

#%cd /content/models/mymodel/
%cd /content/gdrive/MyDrive/Unikin/memoire/roidetection/mymodel/

# Télécharger les poids des modèles pré-entraînés
import tarfile
download_tar = 'http://download.tensorflow.org/models/object_detection/tf2/20200711/' + pretrained_checkpoint
!wget {download_tar}
tar = tarfile.open(pretrained_checkpoint)
tar.extractall()
tar.close()

# Télécharger le fichier de configuration d'entrainement pour le modèle
download_config = 'https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/configs/tf2/' + base_pipeline_file
!wget {download_config}

/content/models/mymodel
--2024-01-11 06:31:37--  http://download.tensorflow.org/models/object_detection/tf2/20200711/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8.tar.gz
Resolving download.tensorflow.org (download.tensorflow.org)... 64.233.183.207, 173.194.193.207, 173.194.194.207, ...
Connecting to download.tensorflow.org (download.tensorflow.org)|64.233.183.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20515344 (20M) [application/x-tar]
Saving to: ‘ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8.tar.gz’

ssd_mobilenet_v2_fp 100%[===================>]  19.56M  --.-KB/s    in 0.1s    

2024-01-11 06:31:37 (138 MB/s) - ‘ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8.tar.gz’ saved [20515344/20515344]

--2024-01-11 06:31:38--  https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/configs/tf2/ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8.config
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.

Maintenant que nous avons téléchargé notre modèle et notre fichier de configuration, nous devons modifier le fichier de configuration avec certains paramètres de formation de haut niveau. Les variables suivantes sont utilisées pour contrôler les étapes de formation :

* **num_steps** : nombre total d'étapes à utiliser pour entraîner le modèle. Un bon chiffre pour commencer est de 40 000 pas. Vous pouvez utiliser plus d'étapes si vous remarquez que les mesures de perte continuent de diminuer à la fin de l'entraînement. Plus il y a d'étapes, plus la formation sera longue. L'entraînement peut également être arrêté prématurément si la perte s'aplatit avant d'atteindre le nombre d'étapes spécifié.
* **batch_size** : le nombre d'images à utiliser par étape de formation. Une taille de lot plus grande permet à un modèle d'être entraîné en moins d'étapes, mais la taille est limitée par la mémoire GPU disponible pour l'entraînement. Avec les GPU utilisés dans les instances Colab, 16 est un bon nombre pour les modèles SSD et 4 est bon pour les modèles EfficientDet.

D'autres informations de formation, telles que l'emplacement du fichier de modèle pré-entraîné, le fichier de configuration et le nombre total de classes, sont également attribuées à cette étape.

In [ ]:
# Définir les paramètres d'entrainement pour le modèle
num_steps = 40000

if chosen_model == 'efficientdet-d0':
  batch_size = 4
else:
  batch_size = 16

In [ ]:
# Définir les emplacements des fichiers et obtenir le nombre de classes pour le fichier de configuration

pipeline_fname = '/content/models/mymodel/' + base_pipeline_file
fine_tune_checkpoint = '/content/models/mymodel/' + model_name + '/checkpoint/ckpt-0'

def get_num_classes(pbtxt_fname):
    from object_detection.utils import label_map_util
    label_map = label_map_util.load_labelmap(pbtxt_fname)
    categories = label_map_util.convert_label_map_to_categories(
        label_map, max_num_classes=90, use_display_name=True)
    category_index = label_map_util.create_category_index(categories)
    return len(category_index.keys())
num_classes = get_num_classes(label_map_pbtxt_fname)
print('Total classes:', num_classes)


Total classes: 1


Ensuite, nous réécrirons le fichier de configuration pour utiliser les paramètres d'entraînement que nous venons de spécifier. La section de code suivante remplacera automatiquement les paramètres nécessaires dans le fichier .config téléchargé et l'enregistrera sous notre fichier personnalisé "pipeline_file.config".

In [ ]:
# Créez un fichier de configuration personnalisé en écrivant l'ensemble de données,
# le point de contrôle du modèle et les paramètres de formation dans le fichier de pipeline de base

import re

%cd /content/models/mymodel
print('writing custom configuration file')

with open(pipeline_fname) as f:
    s = f.read()
with open('pipeline_file.config', 'w') as f:

    # Définir le chemin fine_tune_checkpoint
    s = re.sub('fine_tune_checkpoint: ".*?"',
               'fine_tune_checkpoint: "{}"'.format(fine_tune_checkpoint), s)

    # Définir les fichiers tfrecord pour les ensembles de données d'entraînement et de test
    s = re.sub(
        '(input_path: ".*?)(PATH_TO_BE_CONFIGURED/train)(.*?")', 'input_path: "{}"'.format(train_record_fname), s)
    s = re.sub(
        '(input_path: ".*?)(PATH_TO_BE_CONFIGURED/val)(.*?")', 'input_path: "{}"'.format(val_record_fname), s)

    # Définir label_map_path
    s = re.sub(
        'label_map_path: ".*?"', 'label_map_path: "{}"'.format(label_map_pbtxt_fname), s)

    # définir batch_size
    s = re.sub('batch_size: [0-9]+',
               'batch_size: {}'.format(batch_size), s)

    # Définir les étapes de formation, num_steps
    s = re.sub('num_steps: [0-9]+',
               'num_steps: {}'.format(num_steps), s)

    # Définir le nombre de classes num_classes

    s = re.sub('num_classes: [0-9]+',
               'num_classes: {}'.format(num_classes), s)

    # Changer le fine-tune checkpoint type de "classification" à "detection"
    s = re.sub(
        'fine_tune_checkpoint_type: "classification"', 'fine_tune_checkpoint_type: "{}"'.format('detection'), s)

    # reduire le taux d'apprentissage si l'on utilise ssd-mobilenet-v2 (car il est trop élevé dans le fichier de configuration par défaut)

    if chosen_model == 'ssd-mobilenet-v2':
      s = re.sub('learning_rate_base: .8',
                 'learning_rate_base: .08', s)

      s = re.sub('warmup_learning_rate: 0.13333',
                 'warmup_learning_rate: .026666', s)

    # Si vous utilisez efficientdet-d0, utilisez fix_shape_resizer au lieu de keep_aspect_ratio_resizer (car il n'est pas pris en charge par TFLite)
    if chosen_model == 'efficientdet-d0':
      s = re.sub('keep_aspect_ratio_resizer', 'fixed_shape_resizer', s)
      s = re.sub('pad_to_max_dimension: true', '', s)
      s = re.sub('min_dimension', 'height', s)
      s = re.sub('max_dimension', 'width', s)

    f.write(s)


/content/models/mymodel
writing custom configuration file


In [ ]:
#Afficher le contenu du fichier de configuration personnalisé
!cat /content/models/mymodel/pipeline_file.config

# SSD with Mobilenet v2 FPN-lite (go/fpn-lite) feature extractor, shared box
# predictor and focal loss (a mobile version of Retinanet).
# Retinanet: see Lin et al, https://arxiv.org/abs/1708.02002
# Trained on COCO, initialized from Imagenet classification checkpoint
# Train on TPU-8
#
# Achieves 22.2 mAP on COCO17 Val

model {
  ssd {
    inplace_batchnorm_update: true
    freeze_batchnorm: false
    num_classes: 1
    box_coder {
      faster_rcnn_box_coder {
        y_scale: 10.0
        x_scale: 10.0
        height_scale: 5.0
        width_scale: 5.0
      }
    }
    matcher {
      argmax_matcher {
        matched_threshold: 0.5
        unmatched_threshold: 0.5
        ignore_thresholds: false
        negatives_lower_than_unmatched: true
        force_match_for_each_row: true
        use_matmul_gather: true
      }
    }
    similarity_calculator {
      iou_similarity {
      }
    }
    encode_background_as_zeros: true
    anchor_generator {
      multiscale_anchor_generator {

Enfin, définissons les emplacements du fichier de configuration et du répertoire de sortie du modèle en tant que variables afin de pouvoir les référencer lorsque nous appelons la commande de formation.

In [ ]:
# definir le chemin d'accès au fichier de configuration personnalisé et au répertoire dans lequel stocker les points de contrôle de l'entrainement
pipeline_file = '/content/models/mymodel/pipeline_file.config'
model_dir = '/content/training/'

# 5.&nbsp;Former un modèle de détection TFLite personnalisé

Nous sommes prêts à entraîner notre modèle de détection d'objets ! Avant de commencer l'entraînement, chargeons une session TensorBoard pour suivre la progression de l'entraînement.
en exécutant la section de code suivante, une session TensorBoard apparaîtra dans le navigateur.
Cela ne montrera rien pour l’instant, car nous n’avons pas commencé l’entraînement. Une fois la formation commencée, on pourra revenir et puis actualiser pour voir la perte globale du modèle.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir '/content/training/train'

La formation du modèle est effectuée à l'aide du script "model_main_tf2.py" de l'API TF Object Detection. La formation durera de 2 à 6 heures, selon le modèle, la taille du lot et le nombre d'étapes de formation. Nous avons déjà défini tous les paramètres et arguments utilisés par `model_main_tf2.py` dans les sections précédentes de ce Colab.
le bloc suivant va commencer l'entraînement !



> *Remarque : le programme peut prendre quelques minutes pour afficher les messages d'entraînement, car il n'affiche les journaux qu'une fois toutes les 100 étapes.*

Pour rester connecté à Google Colab même lorsque on est inactif,
nous avons ecrit le script JS suivant pour simuler le clic

le script est exécuté dans la console du navigateur: ```ctrl + shift + i```




```
function ConnectButton() {
    console.log("En cours d'exécution...");
    
    const connectButton = document.querySelector("colab-connect-button").shadowRoot.querySelector('#connect');
    
    if (connectButton) {
        connectButton.click();
    } else {
        console.log("Bouton de connexion non trouvé.");
    }
}

setInterval(ConnectButton, 60000);

```


In [ ]:
# lancer l'entrainement!
!python /content/models/research/object_detection/model_main_tf2.py \
    --pipeline_config_path={pipeline_file} \
    --model_dir={model_dir} \
    --alsologtostderr \
    --num_train_steps={num_steps} \
    --sample_1_of_n_eval_examples=1

/usr/local/lib/python3.10/dist-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/usr/local/lib/python3.10/dist-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.13.0 and strictly below 2.16.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.8.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're us

Si nous souhaitons arrêter l'entraînement plus tôt, il nous faut cliquer sur Arrêter plusieurs fois ou faire un clic droit sur le bloc de code puis sélectionner « Interrompre l'exécution ». Sinon, l’entraînement s’arrêtera automatiquement une fois le nombre d’étapes spécifié atteint.

# 6.&nbsp;Convertir le modèle en TensorFlow Lite

Bien! Notre modèle est entièrement formé et prêt à être utilisé pour détecter des objets. Tout d'abord, nous devons exporter le graphique du modèle (un fichier contenant des informations sur l'architecture et les pondérations) vers un format compatible TensorFlow Lite. Nous ferons cela en utilisant le script `export_tflite_graph_tf2.py`.

In [ ]:
# Make a directory to store the trained TFLite model
!mkdir /content/custom_model_lite
output_directory = '/content/custom_model_lite'

# Path to training directory (the conversion script automatically chooses the highest checkpoint file)
last_model_path = '/content/training'

!python /content/models/research/object_detection/export_tflite_graph_tf2.py \
    --trained_checkpoint_dir {last_model_path} \
    --output_directory {output_directory} \
    --pipeline_config_path {pipeline_file}


Ensuite, nous prendrons le graphique exporté et utiliserons le module `TFLiteConverter` pour le convertir au format `.tflite` FlatBuffer.

In [ ]:
# Convert exported graph file into TFLite model file
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model('/content/custom_model_lite/saved_model')
tflite_model = converter.convert()

with open('/content/custom_model_lite/detect.tflite', 'wb') as f:
  f.write(tflite_model)

# 7. Tester le modèle TensorFlow Lite et calculez mAP

Nous avons formé notre modèle personnalisé et l'avons converti au format TFLite. Mais dans quelle mesure est-il réellement efficace pour détecter des objets dans les images ? C'est là qu'interviennent les images que nous avons réservées dans le dossier **test**. Le modèle n'a jamais vu d'images de test pendant l'entraînement, ses performances sur ces images devraient donc être représentatives de ses performances sur les nouvelles images du terrain.

### 7.1 Images de tests d'inférence
Le code suivant définit une fonction pour exécuter l'inférence sur les images de test. Il charge les images, charge le modèle et le labelmap, exécute le modèle sur chaque image et affiche le résultat. Il enregistre également éventuellement les résultats de détection sous forme de fichiers texte afin que nous puissions les utiliser pour calculer le score mAP du modèle.

https://github.com/EdjeElectronics/TensorFlow-Lite-Object-Detection-on-Android-and-Raspberry-Pi/blob/master/TFLite_detection_image.py

https://github.com/EdjeElectronics/TensorFlow-Lite-Object-Detection-on-Android-and-Raspberry-Pi

In [ ]:
# Script to run custom TFLite model on test images to detect objects
# Source: https://github.com/EdjeElectronics/TensorFlow-Lite-Object-Detection-on-Android-and-Raspberry-Pi/blob/master/TFLite_detection_image.py

# Import packages
import os
import cv2
import numpy as np
import sys
import glob
import random
import importlib.util
from tensorflow.lite.python.interpreter import Interpreter

import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline

### Define function for inferencing with TFLite model and displaying results

def tflite_detect_images(modelpath, imgpath, lblpath, min_conf=0.5, num_test_images=10, savepath='/content/results', txt_only=False):

  # Grab filenames of all images in test folder
  images = glob.glob(imgpath + '/*.jpg') + glob.glob(imgpath + '/*.JPG') + glob.glob(imgpath + '/*.png') + glob.glob(imgpath + '/*.bmp')

  # Load the label map into memory
  with open(lblpath, 'r') as f:
      labels = [line.strip() for line in f.readlines()]

  # Load the Tensorflow Lite model into memory
  interpreter = Interpreter(model_path=modelpath)
  interpreter.allocate_tensors()

  # Get model details
  input_details = interpreter.get_input_details()
  output_details = interpreter.get_output_details()
  height = input_details[0]['shape'][1]
  width = input_details[0]['shape'][2]

  float_input = (input_details[0]['dtype'] == np.float32)

  input_mean = 127.5
  input_std = 127.5

  # Randomly select test images
  images_to_test = random.sample(images, num_test_images)

  # Loop over every image and perform detection
  for image_path in images_to_test:

      # Load image and resize to expected shape [1xHxWx3]
      image = cv2.imread(image_path)
      image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
      imH, imW, _ = image.shape
      image_resized = cv2.resize(image_rgb, (width, height))
      input_data = np.expand_dims(image_resized, axis=0)

      # Normalize pixel values if using a floating model (i.e. if model is non-quantized)
      if float_input:
          input_data = (np.float32(input_data) - input_mean) / input_std

      # Perform the actual detection by running the model with the image as input
      interpreter.set_tensor(input_details[0]['index'],input_data)
      interpreter.invoke()

      # Retrieve detection results
      boxes = interpreter.get_tensor(output_details[1]['index'])[0] # Bounding box coordinates of detected objects
      classes = interpreter.get_tensor(output_details[3]['index'])[0] # Class index of detected objects
      scores = interpreter.get_tensor(output_details[0]['index'])[0] # Confidence of detected objects

      detections = []

      # Loop over all detections and draw detection box if confidence is above minimum threshold
      for i in range(len(scores)):
          if ((scores[i] > min_conf) and (scores[i] <= 1.0)):

              # Get bounding box coordinates and draw box
              # Interpreter can return coordinates that are outside of image dimensions, need to force them to be within image using max() and min()
              ymin = int(max(1,(boxes[i][0] * imH)))
              xmin = int(max(1,(boxes[i][1] * imW)))
              ymax = int(min(imH,(boxes[i][2] * imH)))
              xmax = int(min(imW,(boxes[i][3] * imW)))

              cv2.rectangle(image, (xmin,ymin), (xmax,ymax), (10, 255, 0), 2)

              # Draw label
              object_name = labels[int(classes[i])] # Look up object name from "labels" array using class index
              label = '%s: %d%%' % (object_name, int(scores[i]*100)) # Example: 'person: 72%'
              labelSize, baseLine = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2) # Get font size
              label_ymin = max(ymin, labelSize[1] + 10) # Make sure not to draw label too close to top of window
              cv2.rectangle(image, (xmin, label_ymin-labelSize[1]-10), (xmin+labelSize[0], label_ymin+baseLine-10), (255, 255, 255), cv2.FILLED) # Draw white box to put label text in
              cv2.putText(image, label, (xmin, label_ymin-7), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2) # Draw label text

              detections.append([object_name, scores[i], xmin, ymin, xmax, ymax])


      # All the results have been drawn on the image, now display the image
      if txt_only == False: # "text_only" controls whether we want to display the image results or just save them in .txt files
        image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12,16))
        plt.imshow(image)
        plt.show()

      # Save detection results in .txt files (for calculating mAP)
      elif txt_only == True:

        # Get filenames and paths
        image_fn = os.path.basename(image_path)
        base_fn, ext = os.path.splitext(image_fn)
        txt_result_fn = base_fn +'.txt'
        txt_savepath = os.path.join(savepath, txt_result_fn)

        # Write results to text file
        # (Using format defined by https://github.com/Cartucho/mAP, which will make it easy to calculate mAP)
        with open(txt_savepath,'w') as f:
            for detection in detections:
                f.write('%s %.4f %d %d %d %d\n' % (detection[0], detection[1], detection[2], detection[3], detection[4], detection[5]))

  return

Le bloc suivant définit les chemins d'accès aux images et modèles de test, puis exécute la fonction d'inférence. pour utiliser plus de 10 images, faut modifiez la variable `images_to_test`.

In [ ]:
# Set up variables for running user's model
PATH_TO_IMAGES='/content/images/test'   # Path to test images folder
PATH_TO_MODEL='/content/custom_model_lite/detect.tflite'   # Path to .tflite model file
PATH_TO_LABELS='/content/labelmap.txt'   # Path to labelmap.txt file
min_conf_threshold=0.5   # Confidence threshold (try changing this to 0.01 if you don't see any detection results)
images_to_test = 10   # Number of images to run detection on

# Run inferencing function!
tflite_detect_images(PATH_TO_MODEL, PATH_TO_IMAGES, PATH_TO_LABELS, min_conf_threshold, images_to_test)

### 7.2 Calculer mAP
Nous avons maintenant une idée visuelle des performances de notre modèle sur les images de test, mais comment pouvons-nous mesurer quantitativement sa précision ?

Une méthode populaire pour mesurer la précision du modèle de détection d'objets est la « mean average precision » (mAP). Fondamentalement, plus le score mAP est élevé, meilleur est le modèle pour détecter les objets dans les images. (https://blog.roboflow.com/mean-average-precision/).

Nous utiliserons l'outil de calcul mAP sur https://github.com/Cartucho/mAP pour déterminer le score mAP de notre modèle. Tout d’abord, nous devons cloner le référentiel et supprimer ses exemples de données existants. Nous téléchargerons également un autre script pour l'interface avec la calculatrice.

In [ ]:
%%bash
git clone https://github.com/Cartucho/mAP /content/mAP
cd /content/mAP
rm input/detection-results/*
rm input/ground-truth/*
rm input/images-optional/*
wget https://raw.githubusercontent.com/EdjeElectronics/TensorFlow-Lite-Object-Detection-on-Android-and-Raspberry-Pi/master/util_scripts/calculate_map_cartucho.py

Ensuite, nous copierons les images et les données d'annotation du dossier **test** vers les dossiers appropriés à l'intérieur du référentiel cloné. Celles-ci seront utilisées comme « données de vérité terrain » auxquelles les résultats de détection de notre modèle seront comparés.

In [ ]:
!cp /content/images/test/* /content/mAP/input/images-optional # Copy images and xml files
!mv /content/mAP/input/images-optional/*.xml /content/mAP/input/ground-truth/  # Move xml files to the appropriate folder

L'outil de calcul attend les données d'annotation dans un format différent du format de fichier Pascal VOC .xml que nous utilisons.
Heureusement, il fournit un script simple, « convert_gt_xml.py », pour la conversion au format .txt attendu.

In [ ]:
!python /content/mAP/scripts/extra/convert_gt_xml.py

nous avons configuré les données de vérité terrain, mais nous avons maintenant besoin des résultats de détection réels de notre modèle. Les résultats de détection seront comparés aux données de vérité terrain pour calculer la précision du modèle dans mAP.

La fonction d'inférence que nous avons définie à l'étape 7.1 peut être utilisée pour générer des données de détection pour toutes les images du dossier **test**. Nous l'utiliserons de la même manière qu'auparavant, sauf que cette fois nous lui dirons de sauvegarder les résultats de détection dans le dossier `detection-results`.


In [ ]:
# Set up variables for running inference, this time to get detection results saved as .txt files
PATH_TO_IMAGES='/content/images/test'   # Path to test images folder
PATH_TO_MODEL='/content/custom_model_lite/detect.tflite'   # Path to .tflite model file
PATH_TO_LABELS='/content/labelmap.txt'   # Path to labelmap.txt file
PATH_TO_RESULTS='/content/mAP/input/detection-results' # Folder to save detection results in
min_conf_threshold=0.1   # Confidence threshold

# Use all the images in the test folder
image_list = glob.glob(PATH_TO_IMAGES + '/*.jpg') + glob.glob(PATH_TO_IMAGES + '/*.JPG') + glob.glob(PATH_TO_IMAGES + '/*.png') + glob.glob(PATH_TO_IMAGES + '/*.bmp')
images_to_test = min(500, len(image_list)) # If there are more than 500 images in the folder, just use 500

# Tell function to just save results and not display images
txt_only = True

# Run inferencing function!
print('Starting inference on %d images...' % images_to_test)
tflite_detect_images(PATH_TO_MODEL, PATH_TO_IMAGES, PATH_TO_LABELS, min_conf_threshold, images_to_test, PATH_TO_RESULTS, txt_only)
print('Finished inferencing!')

Enfin, calculons mAP !
une des façons populaires de rapporter le mAP est en utilisant la métrique COCO pour le mAP @ 0,50:0,95. En d'autres termes, cela signifie que le mAP est calculé à plusieurs seuils d'intersection sur union (IoU) compris entre 0,50 et 0,95, puis le résultat de chaque seuil est moyenné pour obtenir un score final de mAP. Le mAP est une mesure de performance couramment utilisée pour évaluer la précision des modèles de détection d'objets dans le domaine de la vision par ordinateur. Il combine à la fois la précision et le rappel pour évaluer la qualité des prédictions du modèle.
 [En savoir plus ici !](https://blog.roboflow.com/mean-average-precision/)

J'ai écrit un script pour exécuter l'outil de calcul à chaque seuil IoU, faire la moyenne des résultats et rapporter le score de précision final. Il rapporte le mAP pour chaque classe et le mAP global.

In [ ]:
%cd /content/mAP
!python calculate_map_cartucho.py --labels=/content/labelmap.txt

Le score indiqué à la fin est le score mAP global de votre modèle. Idéalement, il devrait être supérieur à 50 % (0,50). Si ce n'est pas le cas, on peut augmenter la précision du modèle en ajoutant davantage d'images à l'ensemble de données.


# 8. Déployer le modèle TensorFlow Lite

Maintenant que notre modèle personnalisé a été entraîné et converti au format TFLite, il est prêt à être téléchargé et déployé dans une application ! Cette section montre comment télécharger le modèle et fournit des liens vers des instructions pour le déployer sur le Raspberry Pi ou d'autres appareils périphériques.

## 8.1. Télécharger le modèle TFLite

Exécutez les deux cellules suivantes pour copier les fichiers labelmap dans le dossier modèle, compressez-les dans un dossier zip, puis téléchargez-les. Le dossier zip contient le modèle `detect.tflite` et les fichiers labelmap `labelmap.txt` qui sont nécessaires pour exécuter le modèle dans votre application.

In [ ]:
# Move labelmap and pipeline config files into TFLite model folder and zip it up
!cp /content/labelmap.txt /content/custom_model_lite
!cp /content/labelmap.pbtxt /content/custom_model_lite
!cp /content/models/mymodel/pipeline_file.config /content/custom_model_lite

%cd /content
!zip -r custom_model_lite.zip custom_model_lite

In [ ]:
from google.colab import files

files.download('/content/custom_model_lite.zip')

Le fichier `custom_model_lite.zip` contenant le modèle sera téléchargé dans notre dossier Téléchargements. Il sera pret à être déployé sur un appareil !